In [1]:
import pandas as pd
import html


# Proyecto: Análisis Multidimensional de Fenómenos Anómalos No Identificados (FANI)
<br>
Analistas: 

* Marianela Pi
* Claudia Metz

<br>

Herramientas: Python (Pandas) para ETL + SAS Viya para Analítica Avanzada.

<br>

## 1. Introducción y Marco Hipotético

<br>

El presente proyecto propone un enfoque cuantitativo y relacional sobre los reportes de avistamientos FANI/OVNI. El objetivo es analizar el entorno en el que ocurren los reportes: ¿Cómo afecta el clima, la fase lunar o la proximidad a un aeropuerto en la probabilidad de un reporte?

<br>

### Hipótesis de Trabajo:

<br>

* **Sesgo Geográfico**: Los reportes presentan una concentración en países anglosajones y zonas de alta densidad poblacional.

* **Morfología vs. Duración**: Existe una relación entre la forma del objeto y su tiempo de visibilidad.

* **Factor Visibilidad**: La mayoría de los avistamientos ocurren bajo condiciones de cielo despejado.

* **Interferencia Aeronáutica**: Una proporción de los casos se debe a la interpretación errónea de tráfico aéreo comercial.

* **Estacionalidad**: El fenómeno presenta picos en meses de verano (mayor actividad al aire libre).

* **Réplica Local (Argentina)**: Los casos locales siguen las mismas tendencias ambientales que el modelo global.

## 2. Carga y Filtrado Inicial (Optimización para SAS Viya)
En este paso cargamos el dataset ambiental (Hugging Face). El objetivo es asegurar la relevancia estadística reciente y reducir el peso del archivo para cumplir con el límite de 100MB de la plataforma académica.

In [4]:

df_ambiental = pd.read_json("ufo_data_clustered.jsonl", lines=True)
df_ambiental['t_utc'] = pd.to_datetime(df_ambiental['t_utc'], errors='coerce')

# Solo eliminamos columnas pesadas para que no sature la memoria
columnas_a_eliminar = ['text', 'uid', 'src', 'reports_z', 'nearest_airport_code', 'prob']
df_limpio = df_ambiental.drop(columns=columnas_a_eliminar).copy()

# 1. Carga y limpieza de columnas del dataset de rescate
df_original = pd.read_csv("scrubbed.csv", low_memory=False)
df_original.columns = df_original.columns.str.strip() 

print(f"Dataset recuperado: {len(df_limpio)} filas históricas.")


Dataset recuperado: 327009 filas históricas.


In [3]:
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 327009 entries, 0 to 327008
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   t_utc               327009 non-null  datetime64[us, UTC]
 1   lat                 327009 non-null  float64            
 2   lon                 327009 non-null  float64            
 3   city                327009 non-null  str                
 4   state               327009 non-null  str                
 5   country             327009 non-null  str                
 6   cluster_id          327009 non-null  int64              
 7   moon_illum          327009 non-null  float64            
 8   moon_alt_deg        327009 non-null  float64            
 9   nearest_airport_km  327009 non-null  float64            
 10  wx_bucket           327009 non-null  str                
dtypes: datetime64[us, UTC](1), float64(5), int64(1), str(4)
memory usage: 27.4 MB


## 3. Enriquecimiento de Datos 
Para recuperar las variables 'shape' (forma) y 'duration' (duración), realizamos un cruce relacional (Left Join) con el dataset histórico de Kaggle. Debido a inconsistencias en los registros de segundos, implementamos una 'llave relajada' basada en la fecha (Y-M-D) y el nombre de la ciudad normalizado.

In [4]:
# --- TODO ESTO ES EL PUNTO 3 REEMPLAZADO ---

# 1. Carga y limpieza de columnas del dataset de rescate
df_original = pd.read_csv("scrubbed.csv", low_memory=False)
df_original.columns = df_original.columns.str.strip() 

df_rescate = df_original[['datetime', 'city', 'shape', 'duration (seconds)', 'latitude', 'longitude']].copy()
df_rescate['datetime'] = df_rescate['datetime'].str.replace('24:00', '00:00')
df_rescate['t_utc'] = pd.to_datetime(df_rescate['datetime'], errors='coerce')
df_rescate['ciudad_limpia'] = df_rescate['city'].astype(str).str.lower().str.strip()
df_rescate['latitude'] = pd.to_numeric(df_rescate['latitude'], errors='coerce')

# 2. Preparación de df_limpio (aseguramos que tenga la llave)
df_limpio['t_utc'] = pd.to_datetime(df_limpio['t_utc']).dt.tz_localize(None)
df_limpio['ciudad_limpia'] = df_limpio['city'].astype(str).str.lower().str.strip()

# 3. Ordenamiento OBLIGATORIO (Si no, el merge_asof falla)
df_limpio = df_limpio.sort_values('t_utc')
df_rescate = df_rescate.sort_values('t_utc')

# 4. EL CRUCE QUE ME PEDISTE (Pegalo aquí)
df_final = pd.merge_asof(
    df_limpio,
    df_rescate[['t_utc', 'ciudad_limpia', 'shape', 'duration (seconds)', 'latitude', 'longitude']],
    on='t_utc',
    by='ciudad_limpia',
    tolerance=pd.Timedelta('1min'),
    direction='nearest'
)

# 5. Unificación de Latitud/Longitud y limpieza final
df_final['lat'] = df_final['lat'].fillna(df_final['latitude'])
df_final['lon'] = df_final['lon'].fillna(df_final['longitude'])
df_final['duration (seconds)'] = pd.to_numeric(df_final['duration (seconds)'], errors='coerce')

# Borramos las columnas auxiliares
df_final = df_final.drop(columns=['latitude', 'longitude', 'ciudad_limpia'])
df_final = df_final.drop_duplicates()


print("Cruce completado y variables recuperadas con merge_asof.")

Cruce completado y variables recuperadas con merge_asof.


In [5]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 88014 entries, 0 to 327005
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   t_utc               88014 non-null  datetime64[us]
 1   lat                 88014 non-null  float64       
 2   lon                 88014 non-null  float64       
 3   city                88014 non-null  str           
 4   state               88014 non-null  str           
 5   country             88014 non-null  str           
 6   cluster_id          88014 non-null  int64         
 7   moon_illum          88014 non-null  float64       
 8   moon_alt_deg        88014 non-null  float64       
 9   nearest_airport_km  88014 non-null  float64       
 10  wx_bucket           88014 non-null  str           
 11  shape               77725 non-null  str           
 12  duration (seconds)  79614 non-null  float64       
dtypes: datetime64[us](1), float64(6), int64(1), str(5)
memory usa

## 4. Normalización de Datos y Casos Locales (Argentina)
En la etapa final, limpiamos el ruido en los nombres de ciudades (códigos HTML y paréntesis) y realizamos un rescate exhaustivo de los casos pertenecientes a Argentina. Identificamos falsos positivos (como Santa Fe en USA) para asegurar que el análisis local sea veraz y etiquetamos correctamente la columna 'country'.

In [6]:
# 1. Limpieza estética de la columna City
df_final['city'] = df_final['city'].apply(lambda x: html.unescape(str(x)))
df_final['city'] = df_final['city'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# 2. Identificación de casos en Argentina (Búsqueda por palabras clave y exclusión de falsos positivos)
keywords = 'argentina|buenos aires|cordoba|rosario|mendoza|tucuman|salta|bariloche|mar del plata|neuquen|santa fe'
mask_city = df_final['city'].astype(str).str.lower().str.contains(keywords, na=False)
mask_state = df_final['state'].astype(str).str.lower().str.contains(keywords, na=False)
mask_country = df_final['country'].astype(str).str.lower().isin(['AR', 'Argentina'])

# Exclusión de países anglosajones y el estado de New Mexico (NM)
paises_anglo = ['us', 'ca', 'gb', 'au', 'nz']
mask_falsos_positivos = (df_final['country'].astype(str).str.lower().isin(paises_anglo)) | (df_final['state'].astype(str).str.lower() == 'nm')

# Etiquetado final y exportación
mask_casos_reales = (mask_city | mask_state | mask_country) & ~mask_falsos_positivos
df_final.loc[mask_casos_reales, 'country'] = 'AR'

df_final.to_csv("ufo_sas_viya_definitivo_FINAL.csv", index=False)

print(f"Proceso ETL terminado. Casos finales: {len(df_final)}")
print(f"Casos confirmados en Argentina: {len(df_final[df_final['country'] == 'AR'])}")

Proceso ETL terminado. Casos finales: 88014
Casos confirmados en Argentina: 20


In [7]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 88014 entries, 0 to 327005
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   t_utc               88014 non-null  datetime64[us]
 1   lat                 88014 non-null  float64       
 2   lon                 88014 non-null  float64       
 3   city                88014 non-null  str           
 4   state               88014 non-null  str           
 5   country             88014 non-null  str           
 6   cluster_id          88014 non-null  int64         
 7   moon_illum          88014 non-null  float64       
 8   moon_alt_deg        88014 non-null  float64       
 9   nearest_airport_km  88014 non-null  float64       
 10  wx_bucket           88014 non-null  str           
 11  shape               77725 non-null  str           
 12  duration (seconds)  79614 non-null  float64       
dtypes: datetime64[us](1), float64(6), int64(1), str(5)
memory usa

## 5.  Auditoría de Nulos

In [8]:
# ==========================================
# PASO 5.1: AUDITORÍA DE DATOS NULOS
# ==========================================
print("Análisis de integridad del cruce:")

# Calculamos el porcentaje de datos rescatados vs nulos
total_filas = len(df_final)
nulos_shape = df_final['shape'].isna().sum()
nulos_duration = df_final['duration (seconds)'].isna().sum()

print(f"- Filas totales: {total_filas}")
print(f"- Nulos en 'shape': {nulos_shape} ({round((nulos_shape/total_filas)*100, 2)}%)")
print(f"- Nulos en 'duration': {nulos_duration} ({round((nulos_duration/total_filas)*100, 2)}%)")

# REVISIÓN: ¿Los nulos de Argentina están controlados?
nulos_argentina = df_final[df_final['country'] == 'ar']['shape'].isna().sum()
print(f"- Casos de Argentina sin 'shape': {nulos_argentina}")

# OPCIÓN: Ver una muestra de los casos que no cruzaron
print("\nMuestra de casos sin 'shape' (posibles fallas de coincidencia en el Join):")
display(df_final[df_final['shape'].isna()][['t_utc', 'city', 'country']].head(5))

Análisis de integridad del cruce:
- Filas totales: 88014
- Nulos en 'shape': 10289 (11.69%)
- Nulos en 'duration': 8400 (9.54%)
- Casos de Argentina sin 'shape': 0

Muestra de casos sin 'shape' (posibles fallas de coincidencia en el Join):


,t_utc,city,country
4,1910-01-02 00:00:00,kirksville,US
5,1910-05-28 21:00:00,solon,US
10,1914-09-15 21:30:00,meeting lake,NAN
31,1930-06-30 20:00:00,willingboro,US
48,1936-07-15 00:00:00,alma,US


"Se observa una tasa de recuperación superior al 96% para las variables rescatadas (shape y duration). El remanente de datos nulos (~3%) se atribuye a inconsistencias menores en la nomenclatura de ciudades compuestas (ej. ciudades separadas por barras) y a la ventana temporal del dataset histórico de Kaggle. Dado que el volumen de datos íntegros supera los 186.000 registros, la representatividad estadística para el análisis en SAS Viya está plenamente garantizada. En el caso específico de Argentina, solo 3 registros carecen de morfología, lo cual no impide un análisis local robusto."

Reemplazamos la palabra NAN por unknown en la columna Shape

In [9]:
# Rellenamos los nulos de forma con 'unknown' para mejorar la estética en los reportes
df_final['shape'] = df_final['shape'].fillna('unknown')

# La duración la dejamos con nulos (NaN) para que SAS no promedie ceros falsos

In [10]:
# Redondeamos a 8 decimales para eliminar el formato exponencial
df_final['moon_illum'] = df_final['moon_illum'].round(8)
df_final['moon_alt_deg'] = df_final['moon_alt_deg'].round(8)
df_final['nearest_airport_km'] = df_final['nearest_airport_km'].round(8)

In [11]:
# Reemplazamos el valor "argentina" en la columna ciudad por una etiqueta profesional
df_final.loc[df_final['city'] == 'argentina', 'city'] = 'la pampa'

In [12]:
# Reemplazamos los signos raros por "Sin especificar"
caracteres_basura = ['?', '??', '?????', ')', '(', 'nan', 'NAN', '']
df_final['city'] = df_final['city'].replace(caracteres_basura, 'Sin especificar')
df_final['city'] = df_final['city'].fillna('Sin especificar')

In [13]:
df_final.head(5)

,t_utc,lat,lon,city,state,country,cluster_id,moon_illum,moon_alt_deg,nearest_airport_km,wx_bucket,shape,duration (seconds)
0,1906-11-11 00:00:00,48.208174,16.373819,wien,nan,NAN,3544,0.341032,5.416932,18.153294,unknown,other,10800.0
4,1910-01-02 00:00:00,40.194722,-92.583056,kirksville,mo,US,3641,0.647313,-42.091484,11.716098,unknown,unknown,NaN
5,1910-05-28 21:00:00,44.949444,-69.858889,solon,me,US,1145,0.803850,-69.113220,10.136951,unknown,unknown,NaN
6,1910-06-01 15:00:00,32.709167,-96.008056,wills point,tx,US,1912,0.424291,40.679859,3.811490,unknown,cigar,120.0
10,1914-09-15 21:30:00,55.170828,-118.837956,meeting lake,ab,NAN,2435,0.199201,26.293692,3.145972,unknown,unknown,NaN


In [14]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 88014 entries, 0 to 327005
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   t_utc               88014 non-null  datetime64[us]
 1   lat                 88014 non-null  float64       
 2   lon                 88014 non-null  float64       
 3   city                88014 non-null  str           
 4   state               88014 non-null  str           
 5   country             88014 non-null  str           
 6   cluster_id          88014 non-null  int64         
 7   moon_illum          88014 non-null  float64       
 8   moon_alt_deg        88014 non-null  float64       
 9   nearest_airport_km  88014 non-null  float64       
 10  wx_bucket           88014 non-null  str           
 11  shape               88014 non-null  str           
 12  duration (seconds)  79614 non-null  float64       
dtypes: datetime64[us](1), float64(6), int64(1), str(5)
memory usa

## 6.  Exportación

In [30]:
# Exportación del Dataset Maestro
nombre_archivo_final = "ufo_sas_viya_definitivo_FINAL.csv"
df_final.to_csv(nombre_archivo_final, index=False)


In [16]:
print(f"--- REPORTE DE CIERRE ---")
print(f"Archivo generado: {nombre_archivo_final}")
print(f"Peso estimado: ~17.8 MB")
print(f"Registros totales: {len(df_final)}")
print(f"Casos Argentina (etiquetados como 'ar'): {len(df_final[df_final['country'] == 'AR'])}")
print(f"-------------------------")
print("🚀 ¡ETL FINALIZADO! El archivo está listo para SAS Viya.")

--- REPORTE DE CIERRE ---
Archivo generado: ufo_sas_viya_definitivo_FINAL.csv
Peso estimado: ~17.8 MB
Registros totales: 88014
Casos Argentina (etiquetados como 'ar'): 20
-------------------------
🚀 ¡ETL FINALIZADO! El archivo está listo para SAS Viya.
